# Water-ski CV demo on a Colab GPU

Runs the slow parts of the demo on a GPU:
1. **Re-label** buoys with Grounding DINO on many more frames (every 0.2 s instead of 0.5 s), which takes about 10 min on a T4.
2. **Train** the fast buoy detector (YOLO11s, 1280 px). This version is bigger and more accurate than the CPU one, and takes about 15–25 min.
3. **Benchmark** every model on the GPU to get the time and cost per tool.
4. **SAM 2** tracking (detector-guided and memory modes) at GPU speed.
5. **Full annotated demo** on all test videos.
6. **Save** a results zip to your Google Drive, then share the link back in the chat.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (or better), then *Runtime → Run all*.
If the GitHub repo is private, first add a Colab secret named `GH_TOKEN` (key icon on the left) holding a GitHub token with read access.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import os
try:
    from google.colab import userdata
    tok = userdata.get('GH_TOKEN')
except Exception:
    tok = None
url = f"https://{tok}@github.com/lytvynenkomaksim/CV_test.git" if tok else "https://github.com/lytvynenkomaksim/CV_test.git"
if not os.path.exists('/content/CV_test'):
    !git clone --depth 1 -b main {url} /content/CV_test
%cd /content/CV_test
!git log --oneline | head -3

Cloning into '/content/CV_test'...
remote: Enumerating objects: 802, done.
remote: Counting objects: 100% (802/802), done.
remote: Compressing objects: 100% (689/689), done.
remote: Total 802 (delta 4), reused 792 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (802/802), 271.38 MiB | 49.58 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/CV_test
22a1522 GPU support + Colab notebook; add buoy pseudo-label dataset


In [3]:
%%capture install_log
!apt-get -qq install -y libegl1 libgles2
!pip install -q ultralytics transformers accelerate rtmlib onnxruntime-gpu lap mediapipe "git+https://github.com/ultralytics/CLIP.git"


In [4]:
!bash tools/download_models.sh
!yolo settings weights_dir=/content/CV_test/models > /dev/null
os.environ['HF_HOME'] = '/content/CV_test/models/hf'
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

done. The distilled buoy detector (buoy_yolo11s.pt) is in weights/ in the repo; copy it here:
  cp ../weights/buoy_yolo11*.pt .
2.11.0+cu130 Tesla T4


## 1. Re-label buoys with Grounding DINO (GPU)
Same holdout videos as on CPU (2 Italy clips + the low-quality Greece clip). The YouTube video stays unseen.

In [5]:
!python tools/autolabel_gdino.py --out data/buoy_dataset_gpu --every 0.2 \
    --videos data/videos/front_view/*.mp4 data/videos/drive_extra/*.mp4 \
    --holdout "Vincenzo Marino 6 at 35" "Lucas Cornale 6 at 35" "Aaron_Davis" 2>&1 | tail -3
!for s in train val; do echo $s $(ls data/buoy_dataset_gpu/images/$s | wc -l) images, $(cat data/buoy_dataset_gpu/labels/$s/*.txt | wc -l) buoy boxes; done

ItalyPro_SanGervasioD2_ProWomen__Alexandra_Garcia_6_at_38_off_f00852 train buoys=12 0.5s
ItalyPro_SanGervasioD2_ProWomen__Alexandra_Garcia_6_at_38_off_f00858 train buoys=9 0.6s
ItalyPro_SanGervasioD2_ProWomen__Alexandra_Garcia_6_at_38_off_f00864 train buoys=9 0.6s
train 1258 images, 955 buoy boxes
val 350 images, 163 buoy boxes


## 2. Train the fast buoy detector

In [6]:
!python tools/train_buoy_yolo.py --data data/buoy_dataset_gpu/data.yaml --model yolo11s.pt \
    --epochs 80 --imgsz 1280 --batch 16 --workers 4 2>&1 | grep -E "all|saved|Epoch" | tail -5

Results saved to /content/CV_test/runs/train/buoy
saved /content/CV_test/models/buoy_yolo11s.pt
Installed 3 packages in 30ms
ONNX: export success ✅ 4.6s, saved as '/content/CV_test/models/buoy_yolo11s.onnx' (36.7 MB)
Results saved to /content/CV_test/models/buoy_yolo11s.onnx


In [7]:
import pandas as pd
r = pd.read_csv('runs/train/buoy/results.csv'); r.columns = [c.strip() for c in r.columns]
r[['epoch','metrics/precision(B)','metrics/recall(B)','metrics/mAP50(B)','metrics/mAP50-95(B)']].tail(5)

,epoch,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B)
75,76,0.76406,0.71523,0.74468,0.60233
76,77,0.73128,0.71166,0.74122,0.59625
77,78,0.73494,0.71166,0.73867,0.58706
78,79,0.71726,0.71779,0.73496,0.58345
79,80,0.71185,0.71779,0.73568,0.58573


## 3. Benchmark every model on the GPU (low / medium / high quality + YouTube)

In [8]:
!python tools/bench.py \
  data/videos/drive_extra/GreecePro_MenRd3__Aaron_Davis_Rd3_6at12.mp4 \
  "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4" \
  data/videos/drive_extra/ItalyPro_OpenWomenRd2__Julie_Mishler_6at32_Rd2.mp4 \
  data/videos/youtube/youtube_regina_jaquess.mp4 \
  --frames 30 --out data/benchmarks/bench_gpu.json 2>&1 | grep -v -i warn | tail -40

ItalyPro_OpenWomenRd2__Julie_Mishler_6at buoy:grounding-dino-t {'load_s': 1.71, 'ms_per_frame': 564.5, 'p90_ms': 629.5, 'n': 30, 'mean_buoys_per_frame': 1.07, 'frames_with_buoy': 0.67}
Loading weights: 100%|██████████| 418/418 [00:00<00:00, 4909.38it/s]
ItalyPro_OpenWomenRd2__Julie_Mishler_6at buoy:owlv2-b {'load_s': 1.04, 'ms_per_frame': 757.4, 'p90_ms': 923.7, 'n': 30, 'mean_buoys_per_frame': 1.33, 'frames_with_buoy': 0.73}
ItalyPro_OpenWomenRd2__Julie_Mishler_6at pose:yolo-n {'load_s': 0.08, 'ms_per_frame': 15.2, 'p90_ms': 21.0, 'n': 27, 'mean_kpt_conf': 0.783}
ItalyPro_OpenWomenRd2__Julie_Mishler_6at pose:yolo-s {'load_s': 0.11, 'ms_per_frame': 23.2, 'p90_ms': 38.3, 'n': 27, 'mean_kpt_conf': 0.838}
ItalyPro_OpenWomenRd2__Julie_Mishler_6at pose:yolo-m {'load_s': 0.1, 'ms_per_frame': 14.5, 'p90_ms': 21.6, 'n': 27, 'mean_kpt_conf': 0.826}
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4520.95it/s]
ItalyPro_OpenWomenRd2__Julie_Mishler_6at pose:vitpose {'load_s': 0.64, 'ms_per_

## 4. SAM 2 at GPU speed (detector-guided vs memory, tiny/small/base)

In [9]:
V = "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4"
for m in ['sam2.1_t.pt', 'sam2.1_s.pt', 'sam2.1_b.pt']:
    !python tools/sam2_track.py "{V}" --start 9 --dur 5 --model {m} --mode guided --out runs/sam2_gpu 2>&1 | tail -1
!python tools/sam2_track.py "{V}" --start 9 --dur 5 --model sam2.1_t.pt --mode memory --out runs/sam2_gpu 2>&1 | tail -1

{"video": "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4", "start": 9.0, "dur": 5.0, "mode": "guided", "model": "sam2.1_t.pt", "imgsz": 1024, "frames": 150, "buoy_boxes": 308, "buoy_masks_accepted": 163, "buoy_hidden": 145, "skier_masks": 115, "ms_per_frame_detect_track": 47.7, "ms_per_frame_sam": 151.6, "total_s": 37.7, "x_realtime": 7.5}
{"video": "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4", "start": 9.0, "dur": 5.0, "mode": "guided", "model": "sam2.1_s.pt", "imgsz": 1024, "frames": 150, "buoy_boxes": 308, "buoy_masks_accepted": 172, "buoy_hidden": 136, "skier_masks": 115, "ms_per_frame_detect_track": 47.5, "ms_per_frame_sam": 179.6, "total_s": 41.1, "x_realtime": 8.2}
{"video": "data/videos/front_view/Vincenzo Marino 6 at 32 off.mp4", "start": 9.0, "dur": 5.0, "mode": "guided", "model": "sam2.1_b.pt", "imgsz": 1024, "frames": 150, "buoy_boxes": 308, "buoy_masks_accepted": 174, "buoy_hidden": 134, "skier_masks": 115, "ms_per_frame_detect_track": 48.1, "ms_per_fr

## 5. Full annotated demo on all test videos

In [10]:
!python tools/run_all.py --buoy yolo-buoy --pose yolo-s --out runs/demo_gpu 2>&1 | grep -v -i warn | tail -25

  frame 3700/3934  14.08 fps
  frame 3750/3934  14.13 fps
  frame 3800/3934  14.16 fps
  frame 3850/3934  14.18 fps
  frame 3900/3934  14.21 fps
[youtube_regina_jaquess] done: {'turn_buoys_seen': 4, 'turn_buoys_outside': 3, 'score': 2}  CPU processing 66 ms/frame (2.0x real-time), buoy detector every 1 frame(s)
# Demo run: buoy=yolo-buoy, pose=yolo-s, skier=yolo11s, buoy detector every 1 frame(s)

| video | official | score | turn_buoys_seen | turn_buoys_outside | skier_visible | pose_good | buoy_frames | buoy_hidden | hidden_gap_s | ms_per_frame | x_realtime | output |
|---|---|---|---|---|---|---|---|---|---|---|---|---|
| data/videos/drive_extra/GreecePro_MenRd3__Aaron_Davis_Rd3_6at12.mp4 | 6 | 6 | 6 | 6 | 0.943 | 0.895 | 0.125 | 0.233 | 0.53 | 65 | 1.96 | GreecePro_MenRd3__Aaron_Davis_Rd3_6at12.mp4 |
| data/videos/drive_extra/GreecePro_WomenRd3__Allie_Nicholson_Rd3_6at12m.mp4 | 6 | 0 | 5 | 4 | 0.876 | 0.858 | 0.173 | 0.224 | 0.2 | 61 | 3.68 | GreecePro_WomenRd3__Allie_Nicholson_Rd3

## 6. Save results to Google Drive
Afterwards, open `MyDrive/cv_test_results/`, share the zip (*Anyone with the link → Viewer*) and paste the link in the chat.

In [12]:
!mkdir -p results && cp models/buoy_yolo11s.pt models/buoy_yolo11s.onnx results/ 2>/dev/null; \
 cp -r runs/train/buoy results/train_buoy && rm -rf results/train_buoy/weights; \
 cp -r data/benchmarks runs/sam2_gpu runs/demo_gpu results/; \
 cp -r data/buoy_dataset_gpu results/ ; rm -f results/buoy_dataset_gpu/data_resolved.yaml
!cd results && zip -qr ../cv_test_results.zip . && cd .. && ls -lh cv_test_results.zip
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cv_test_results && cp cv_test_results.zip /content/drive/MyDrive/cv_test_results/
print('saved to MyDrive/cv_test_results/cv_test_results.zip')

-rw-r--r-- 1 root root 412M Oct  7 20:32 cv_test_results.zip
Mounted at /content/drive
saved to MyDrive/cv_test_results/cv_test_results.zip
